# ¿Sirven las features de DINOv3 fuera de ImageNet?
### Análisis por capa de un backbone congelado sobre terreno marciano (AI4Mars)

**INF659 Deep Learning · Escuela de Posgrado, Maestría en Informática · 2026-2 · Entrega E1**
**Grupo 4** · **Integrantes:** *[nombre y código de cada integrante]*

---

## Resumen

Los modelos fundacionales auto-supervisados como DINOv3 se evalúan sobre todo en imágenes naturales. Las imágenes de Marte tienen otra estadística (poca variedad de color, texturas repetitivas, iluminación particular y sensores propios), y su aplicación, la navegación autónoma de rovers, es un caso donde el etiquetado experto es costoso y las muestras etiquetadas son escasas.

**Pregunta principal.** ¿Qué tan bien transfieren las features congeladas de DINOv3 a un dominio muy alejado de sus datos de preentrenamiento (terreno de Marte)?

**Preguntas secundarias.** (1) ¿Qué capas aportan las mejores features para este dominio? (2) ¿Cuánto mejora el fine-tuning parcial frente al *linear probing*, y a qué costo?

**Este cuaderno** es la intervención preliminar y responde la pregunta secundaria 1. Se **congela** DINOv3 ViT-B/16, se leen las características de siete profundidades (bloques 2, 4, 6, 8, 10, 12 y la salida final normalizada, `last_norm`) y sobre cada una se entrena **la misma cabeza lineal mínima** sobre los *patch tokens*. Lo único que cambia entre experimentos es la capa de origen; por eso la diferencia de rendimiento refleja la información que esa capa ya contiene.

### Alcance respecto de la propuesta

| Componente de la propuesta | Estado en esta entrega |
|---|---|
| 6.1 *Linear probing*: cabeza 1×1 sobre los patch tokens, logits interpolados a la máscara | Incluido |
| 6.3 Análisis por capa: una cabeza lineal por bloque, mIoU frente a profundidad | Incluido (7 profundidades) |
| §7 Análisis cualitativo: PCA de features, mapas de predicción y casos de fallo | Incluido |
| 6.2 Fine-tuning parcial · 6.4 Eficiencia por muestra etiquetada | Entrega 3 |
| Comparación con DINOv3 `sat493m`, DINOv2 y cota inferior · hipótesis H2 y H3 | Entrega 3 |
| Media ± desviación estándar sobre varias semillas | Entrega 3 (aquí: una corrida con una semilla) |

Se trata de una intervención **preliminar**: las conclusiones se formulan como «en esta corrida» y la sección 11 detalla las limitaciones.

## 1. Configuración

Los parámetros del experimento están reunidos en `CFG`: modelo, tamaño de entrada (448 px, múltiplo del parche de 16 px), tamaño del subconjunto de entrenamiento (1 200 de ≈ 16 000 imágenes, por costo de cómputo), hiperparámetros de la cabeza y la semilla. `tol` es la **diferencia mínima de mIoU que se considera relevante** (0,01, del orden de la variación entre semillas observada en pruebas piloto); se usa en la sección 8.

In [ ]:
import os, sys, json, time, math, random
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import transformers
from PIL import Image
from IPython.display import display, Markdown

CFG = dict(rover="msl", model_id="facebook/dinov3-vitb16-pretrain-lvd1689m", size=448, n_train=1200,
           epochs=20, bs=32, lr=1e-3, weight_decay=1e-4, seed=0, tol=0.01)

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"
os.makedirs(OUT_DIR, exist_ok=True)
DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
RS = Image.Resampling
NUM_WORKERS = min(2, os.cpu_count() or 1)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "axes.titlesize": 11})
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 200)

def log(msg):   # registro silencioso con hora (cuaderno de decisiones: design_log.md)
    with open(os.path.join(OUT_DIR, "design_log.md"), "a", encoding="utf-8") as f: f.write(f"- {time.strftime('%Y-%m-%d %H:%M:%S')} · {msg}\n")

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
assert torch.cuda.is_available(), "Se requiere GPU (en Kaggle: Settings → Accelerator → GPU T4)."
env = dict(torch=torch.__version__, transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(dict(env=env, cfg=CFG), open(os.path.join(OUT_DIR, "env.json"), "w"), indent=2)
log(f"Entorno {env}. Configuración {CFG}")
print(f"Entorno: torch {env['torch']} · transformers {env['transformers']} · {env['gpu']}")

## 2. Datos: AI4Mars (rover Curiosity)

**AI4Mars** (Swan et al., 2021) reúne imágenes del terreno de Marte tomadas por rovers de la NASA, con etiquetas de terreno por píxel producidas por *crowdsourcing*. La tarea es **segmentación semántica** en cuatro clases (suelo, roca madre, arena y roca grande); los píxeles sin etiqueta (`255`) **se ignoran** al entrenar y evaluar. Se usa el espejo de Kaggle (v0.6) del archivo de Zenodo, restringido al rover Curiosity (`msl`).

**Particularidades del dominio y del conjunto**
- **Imágenes:** las cámaras de navegación (NavCam) son en escala de grises; se replican a 3 canales para el modelo.
- **Máscaras:** según la documentación del conjunto, las etiquetas ya excluyen el cuerpo del rover y los puntos a más de 30 m (quedan sin etiqueta).
- **Ruido de etiquetas:** el etiquetado colaborativo limita el techo de mIoU.

**Particiones**
- **Entrenamiento:** etiquetas colaborativas fusionadas (≥ 3 anotadores y 2/3 de acuerdo por píxel).
- **Prueba:** partición oficial con etiquetas de **expertos**, versión `masked-gold-min3-100agree` (las tres personas deben coincidir en cada píxel: la más estricta y dispersa). No interviene en ningún ajuste.

**Preprocesamiento:** entrada de 448 × 448 (múltiplo de 16, el parche de DINOv3), normalización de ImageNet y máscaras redimensionadas por vecino más cercano (no inventa clases). Cada imagen se empareja con su máscara por nombre de archivo y se sortea el subconjunto de entrenamiento.

In [ ]:
INPUT = Path("/kaggle/input")
def find_root():
    for dp, dns, fns in os.walk(INPUT):
        if "label_keys.json" in fns: return Path(dp)
        depth = len(Path(dp).relative_to(INPUT).parts)
        if depth == 0: dns[:] = [d for d in dns if d != "notebooks"]
        if depth >= 5: dns[:] = []
    raise FileNotFoundError("Dataset no encontrado: adjunta 'AI4Mars - Mirrored Dataset' como Input del notebook.")

ROOT = find_root(); R = ROOT / CFG["rover"]
IMG_EXT = (".jpg", ".jpeg", ".png")
dirs = {dp: n for dp, _, fns in os.walk(R) if (n := sum(f.lower().endswith(IMG_EXT) for f in fns))}
rel = lambda p: Path(p).relative_to(R).as_posix().lower()
def pick(cands, prefer=()):
    for kw in prefer: cands = [d for d in cands if kw in rel(d)] or cands
    return max(cands, key=dirs.get) if cands else None

IMG_DIR   = pick([d for d in dirs if "label" not in rel(d) and "edr" in rel(d)])
TRAIN_LBL = pick([d for d in dirs if "label" in rel(d) and "train" in rel(d)])
TEST_LBL  = pick([d for d in dirs if "label" in rel(d) and "test" in rel(d)], prefer=("min3", "100agree"))
assert IMG_DIR and TRAIN_LBL and TEST_LBL, f"No se ubicaron las carpetas esperadas. Carpetas con imágenes: {[rel(d) for d in dirs]}"

IMG_IDX = {Path(f).stem.lower(): os.path.join(IMG_DIR, f) for f in os.listdir(IMG_DIR) if f.lower().endswith(IMG_EXT)}
def make_pairs(lbl_dir):
    pairs = []
    for f in sorted(os.listdir(lbl_dir)):
        if not f.lower().endswith(".png"): continue
        stem = Path(f).stem.lower(); ip = IMG_IDX.get(stem) or IMG_IDX.get(stem.removesuffix("_merged"))
        if ip: pairs.append((ip, os.path.join(lbl_dir, f)))
    return pairs

ALL_TR, TE = make_pairs(TRAIN_LBL), make_pairs(TEST_LBL)
assert ALL_TR and TE, "Sin pares imagen-máscara: revisar la correspondencia de nombres de archivo."
TR = sorted(random.Random(CFG["seed"]).sample(ALL_TR, min(CFG["n_train"], len(ALL_TR))))

CLASS_NAMES = ["suelo", "roca madre", "arena", "roca grande"]; K = len(CLASS_NAMES); IGNORE = 255
LUT = np.full(256, IGNORE, np.uint8); LUT[:K] = np.arange(K)          # valor de máscara → índice de clase
def read_mask(p):
    m = np.array(Image.open(p)); assert m.ndim == 2, f"máscara inesperada: {p}"; return m

print(f"AI4Mars · rover {CFG['rover'].upper()}: {len(ALL_TR):,} pares de entrenamiento disponibles (se usan {len(TR):,}) y {len(TE)} de prueba.")
log(f"Datos AI4Mars/{CFG['rover']}: {len(ALL_TR)} pares train, {len(TE)} test; subconjunto {len(TR)}. IMG={rel(IMG_DIR)}, TRAIN={rel(TRAIN_LBL)}, TEST={rel(TEST_LBL)}.")

## 3. Exploración de los datos

La figura muestra imágenes de entrenamiento con su máscara (gris = sin etiqueta) y el gráfico indica el peso de cada clase entre los píxeles etiquetados. Dos hechos condicionan todo el análisis posterior:

- La **roca grande** es una clase muy escasa.
- La **prueba tiene un reparto de clases distinto al del entrenamiento**, además de etiquetas de otro origen (expertos frente a colaborativas).

Por eso la métrica principal será el **mIoU**, que pesa igual a las cuatro clases, y no la exactitud por píxel, que premia a las clases abundantes.

In [ ]:
COLORS = np.array([(0.55, 0.35, 0.15), (0.30, 0.50, 0.80), (0.95, 0.85, 0.30), (0.85, 0.15, 0.15)])
def colorize(m):
    rgb = np.full(m.shape + (3,), 0.15, np.float32)
    for c in range(K): rgb[m == c] = COLORS[c]
    return rgb

from matplotlib.patches import Patch
CLASS_LEGEND = [Patch(color=COLORS[c], label=CLASS_NAMES[c]) for c in range(K)] + [Patch(color=(0.15,) * 3, label="sin etiqueta")]

def show_samples(pairs, n=3, seed=0, title=""):
    sel = random.Random(seed).sample(pairs, n)
    fig, ax = plt.subplots(n, 3, figsize=(11, 3.6 * n), squeeze=False)
    for r, (ip, mp) in enumerate(sel):
        m = LUT[read_mask(mp).astype(np.intp)]
        img = np.asarray(Image.open(ip).convert("RGB").resize(m.shape[::-1], RS.BILINEAR))
        lab = m != IGNORE; ov = img.astype(np.float32) / 255; ov[lab] = 0.5 * ov[lab] + 0.5 * colorize(m)[lab]
        for a, im_, t in zip(ax[r], (img, colorize(m), ov), ("imagen", f"máscara ({100 * lab.mean():.0f} % etiquetado)", "superposición")):
            a.imshow(im_, interpolation="nearest"); a.set_title(t, fontsize=9); a.axis("off")
    fig.legend(handles=CLASS_LEGEND, loc="lower center", ncol=5, frameon=False)
    fig.suptitle(title); plt.tight_layout(rect=(0, 0.05, 1, 0.97)); plt.show()

def class_share(pairs, n=300):
    cnt = np.zeros(256, np.int64)
    for _, mp in random.Random(1).sample(pairs, min(n, len(pairs))): cnt += np.bincount(read_mask(mp).ravel(), minlength=256)[:256]
    assert set(np.nonzero(cnt)[0]) <= {0, 1, 2, 3, 255}, "Valores de máscara inesperados."
    return cnt[:K] / cnt[:K].sum()

show_samples(TR, 3, 0, "Ejemplos de entrenamiento")
sh_tr, sh_te = class_share(TR), class_share(TE)
fig, ax = plt.subplots(figsize=(7, 3.2)); x = np.arange(K)
ax.bar(x - 0.2, 100 * sh_tr, 0.4, label="entrenamiento"); ax.bar(x + 0.2, 100 * sh_te, 0.4, label="prueba")
for i in range(K):
    ax.text(i - 0.2, 100 * sh_tr[i] + 1, f"{100 * sh_tr[i]:.1f}", ha="center", fontsize=8); ax.text(i + 0.2, 100 * sh_te[i] + 1, f"{100 * sh_te[i]:.1f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES); ax.set_ylabel("% de píxeles etiquetados"); ax.set_title("Balance de clases"); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "balance_clases.png"), dpi=140); plt.show()
log(f"Balance de clases (% píxeles etiquetados) train {dict(zip(CLASS_NAMES, np.round(100 * sh_tr, 1).tolist()))}; test {dict(zip(CLASS_NAMES, np.round(100 * sh_te, 1).tolist()))}.")

## 4. Modelo: DINOv3 ViT-B/16 congelado

Se usa **DINOv3 ViT-B/16** (`lvd1689m`, preentrenado con imágenes web): ≈ 86 M de parámetros, 12 bloques y vectores de 768 números, con todos sus pesos **congelados**.

La imagen de 448 × 448 se divide en parches de 16 × 16 píxeles, es decir, una grilla de **28 × 28 = 784 parches**, y cada parche es un vector (*token*). Antes de ellos van 5 tokens especiales (clase y registros) que se descartan: solo los parches sirven para segmentar.

Las capas intermedias se leen mediante *forward hooks* sobre los bloques, que se localizan por estructura (la lista de módulos con tantos elementos como bloques). Se distingue la **salida del bloque k** de **`last_norm`**, la salida estándar del modelo (bloque 12 más una normalización final); el código verifica que los hooks la reproducen exactamente.

In [ ]:
from transformers import AutoModel
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    backbone = AutoModel.from_pretrained(CFG["model_id"], token=HF_TOKEN).to(DEV).eval()
except Exception as e:
    raise RuntimeError(f"No se pudo cargar {CFG['model_id']}: se requiere un token de Hugging Face (secreto HF_TOKEN, activado en este notebook) "
                       f"con acceso aprobado a ese modelo. Detalle: {e}") from e
for p in backbone.parameters(): p.requires_grad = False

def find_blocks(m):
    # lista de bloques del transformer: el nn.ModuleList con num_hidden_layers elementos y más parámetros
    cands = [mod for _, mod in m.named_modules() if isinstance(mod, nn.ModuleList) and len(mod) == m.config.num_hidden_layers]
    assert cands, "No se encontró la lista de bloques del transformer."
    return max(cands, key=lambda ml: sum(p.numel() for p in ml.parameters()))
BLOCKS = find_blocks(backbone); L = len(BLOCKS)
PS, C, S = backbone.config.patch_size, backbone.config.hidden_size, CFG["size"]
assert S % PS == 0; G = S // PS
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def load_img(path):
    img = Image.open(path).convert("RGB").resize((S, S), RS.BILINEAR)
    return (torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255 - MEAN) / STD

def forward_feats(x, layers):
    # {capa: tokens (B, T, C)}; capa = entero (salida del bloque) o "last_norm"
    buf, hooks = {}, []
    for l in layers:
        if l != "last_norm":
            hooks.append(BLOCKS[l - 1].register_forward_hook(lambda m, i, o, l=l: buf.__setitem__(l, o[0] if isinstance(o, (tuple, list)) else o)))
    try:
        with torch.no_grad(): out = backbone(pixel_values=x)
    finally:
        for h in hooks: h.remove()
    if "last_norm" in layers: buf["last_norm"] = out.last_hidden_state
    return buf

def patch_tokens(h): return h[:, -G * G:, :]          # los tokens especiales van delante

LAYERS = [2, 4, 6, 8, 10, 12, "last_norm"]            # ≈ 1/6, 1/3, 1/2, 2/3, 5/6 y final de la profundidad
PRIMARY = 8                                           # bloque ≈ 2/3 de profundidad
chk = forward_feats(torch.randn(1, 3, S, S, device=DEV), [L, "last_norm"])
final_norm = next(getattr(backbone, n) for n in ("norm", "layernorm") if hasattr(backbone, n))
assert torch.allclose(final_norm(chk[L]), chk["last_norm"], atol=1e-3), "Los hooks no reproducen last_hidden_state."
SEQ = chk["last_norm"].shape[1]; n_params = sum(p.numel() for p in backbone.parameters())
FL_BLOCK = 2 * (4 * SEQ * C * C + 2 * SEQ * SEQ * C + 2 * SEQ * C * getattr(backbone.config, "intermediate_size", 4 * C)) / 1e9   # GFLOPs por bloque (analítico)
print(f"{CFG['model_id']} · {n_params / 1e6:.1f} M parámetros · {L} bloques · {C} dimensiones · grilla {G}×{G} · ≈ {FL_BLOCK:.1f} GFLOPs por bloque")
log(f"Modelo {CFG['model_id']} congelado; {n_params / 1e6:.1f} M parámetros; {L} bloques; {SEQ - G * G} tokens especiales; capas sondeadas {LAYERS}.")

### Qué representa cada capa

Para dos imágenes de prueba se proyectan los 384 números de cada parche a **3 componentes principales (PCA)** y se muestran como color: parches con colores parecidos tienen características parecidas. Es una ilustración cualitativa de cómo evoluciona la representación con la profundidad; no constituye evidencia por sí sola.

In [ ]:
def pca_rgb(t):
    t = t.float() - t.float().mean(0, keepdim=True)
    _, _, V = torch.pca_lowrank(t, q=3, center=False, niter=4)
    comp = (t @ V[:, :3]).cpu().numpy()
    lo, hi = np.percentile(comp, 2, axis=0), np.percentile(comp, 98, axis=0)
    return np.clip((comp - lo) / np.maximum(hi - lo, 1e-6), 0, 1).reshape(G, G, 3)

VIS = random.Random(2).sample(TE, 2)
fig, ax = plt.subplots(len(VIS), 1 + len(LAYERS), figsize=(2.4 * (1 + len(LAYERS)), 2.5 * len(VIS)), squeeze=False)
for r, (ip, _) in enumerate(VIS):
    x = load_img(ip)[None].to(DEV); fe = forward_feats(x, LAYERS)
    ax[r, 0].imshow((x[0].cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()); ax[r, 0].set_title("imagen" if r == 0 else "", fontsize=9)
    for c, l in enumerate(LAYERS, 1):
        ax[r, c].imshow(pca_rgb(patch_tokens(fe[l])[0]), interpolation="nearest")
        if r == 0: ax[r, c].set_title("last_norm" if l == "last_norm" else f"bloque {l}", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.suptitle("PCA de las características de cada capa (3 componentes → RGB)"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pca_capas.png"), dpi=130); plt.show()

## 5. Hipótesis

Cada hipótesis establece qué se espera, por qué, y la regla con la que se juzgará; las reglas usan la tolerancia `tol` = 0,01 de mIoU. La **H1** es la de la propuesta; las **H2 y H3** de la propuesta (brecha entre *linear probing* y fine-tuning según los datos; DINOv3 frente a DINOv2) requieren experimentos de la Entrega 3 y **no se evalúan aquí**. Para esta intervención se añaden dos hipótesis derivadas (H1a, H1b) y el criterio de evaluación.

| | Hipótesis | Razón | Regla |
|---|---|---|---|
| **H1** | Las capas intermedias **superan** a la última capa en este dominio. | La textura local, que distingue arena, suelo y roca, se diluye en las capas finales, orientadas a información global y semántica de imágenes web; en Marte esa semántica transfiere peor que la textura. | Se sostiene si algún bloque intermedio supera a `last_norm` por más de `tol` en mIoU; no se sostiene si `last_norm` supera a todos por más de `tol`; en otro caso, no concluyente. |
| **H1a** | El mIoU **sube con la profundidad y luego se estanca**. | Cada bloque refina lo anterior hasta que queda poca información nueva. | Se sostiene si el mIoU no alcanza su máximo (± `tol`) en la primera capa y, desde que lo alcanza, no cae más de `tol`. |
| **H1b** | La **roca grande** es la clase más difícil en todas las capas. | Es la clase minoritaria (≈ 1 % de los píxeles de entrenamiento): hay pocos ejemplos para aprenderla. | Se sostiene si su IoU es el menor de las cuatro clases en cada capa. |
| **Criterio** | Una capa es «mejor» si lo es en **mIoU**. | El mIoU da el mismo peso a las clases raras y a las comunes. | Se reportan además la exactitud y el IoU por clase. |

In [ ]:
HIPOTESIS = (f"H1 (propuesta): las capas intermedias superan a la última capa (tolerancia {CFG['tol']} de mIoU). "
             f"H1a: el mIoU sube con la profundidad y luego se estanca. H1b: la roca grande es la clase más difícil en todas las capas. "
             f"Criterio: mIoU (clases raras y comunes con igual peso). H2 y H3 de la propuesta quedan para la Entrega 3.")
log("Hipótesis registradas antes de la extracción y el entrenamiento: " + HIPOTESIS)

## 6. Intervención: leer cada capa y entrenar la misma cabeza

**Extracción.** Cada imagen atraviesa DINOv3 una sola vez y se conservan los mapas de parches de las 7 capas (fp16). El backbone no vuelve a ejecutarse.

**Cabeza.** `BatchNorm2d` (iguala las escalas entre capas, de modo que la comparación no dependa de qué capa tiene valores más grandes) seguida de una **convolución 1×1**, es decir, un clasificador lineal de 4 clases por parche; las predicciones se amplían (interpolación bilineal) hasta la resolución de la máscara. Es mínima a propósito: así el resultado refleja lo que la capa ya sabe y no la capacidad de la cabeza.

**Condiciones idénticas para todas las capas:** mismas imágenes, mismo orden de lotes, mismas épocas y tasa de aprendizaje (fijadas de antemano, sin ajustarlas con la prueba) y entropía cruzada que **ignora los píxeles sin etiqueta**. No se aplican aumentos de datos, porque las características se almacenan.

**Referencia mínima:** un clasificador que responde siempre la clase más frecuente del entrenamiento.

In [ ]:
class SegData(torch.utils.data.Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        y = Image.fromarray(LUT[read_mask(mp).astype(np.intp)]).resize((S, S), RS.NEAREST)   # vecino más cercano: no inventa clases
        return load_img(ip), torch.from_numpy(np.asarray(y).copy())

@torch.no_grad()
def extract_all(pairs, bs=16):
    N = len(pairs); feats = {l: torch.empty((N, C, G, G), dtype=torch.float16) for l in LAYERS}; Y = torch.empty((N, S, S), dtype=torch.uint8); i = 0
    for x, y in torch.utils.data.DataLoader(SegData(pairs), batch_size=bs, num_workers=NUM_WORKERS):
        with torch.autocast("cuda", dtype=torch.float16): out = forward_feats(x.to(DEV), LAYERS)
        for l in LAYERS:
            h = patch_tokens(out[l]).float().half()
            assert torch.isfinite(h).all(), f"valores no finitos en la capa {l} (desbordamiento fp16)"
            feats[l][i:i + len(x)] = h.transpose(1, 2).reshape(len(x), C, G, G).cpu()
        Y[i:i + len(x)] = y; i += len(x)
    return feats, Y

t0 = time.time(); Xtr, Ytr = extract_all(TR); Xte, Yte = extract_all(TE)
print(f"Características extraídas: {len(TR)} imágenes de entrenamiento y {len(TE)} de prueba, {len(LAYERS)} capas ({time.time() - t0:.0f} s).")
log(f"Características extraídas una sola vez ({len(TR)} train, {len(TE)} test, capas {LAYERS}); forma por capa {tuple(Xtr[2].shape)}.")

In [ ]:
class Head(nn.Module):
    def __init__(self, c, k): super().__init__(); self.bn = nn.BatchNorm2d(c); self.conv = nn.Conv2d(c, k, 1)
    def forward(self, x): return self.conv(self.bn(x))
def up(z): return F.interpolate(z, size=(S, S), mode="bilinear", align_corners=False)

def train_head(X, Y, layer):
    seed = CFG["seed"]; torch.manual_seed(seed * 100003 + (999 if layer == "last_norm" else layer))
    head = Head(C, K).to(DEV); opt = torch.optim.AdamW(head.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = len(X) // CFG["bs"]; total = CFG["epochs"] * steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * s / total)))
    g = torch.Generator().manual_seed(seed)                      # mismo orden de lotes para todas las capas
    for _ in range(CFG["epochs"]):
        head.train(); perm = torch.randperm(len(X), generator=g)
        for b in range(steps):
            idx = perm[b * CFG["bs"]:(b + 1) * CFG["bs"]].to(DEV); y = Y[idx].long()
            if (y != IGNORE).sum() == 0: continue
            loss = F.cross_entropy(up(head(X[idx].float())), y, ignore_index=IGNORE)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    return head.eval()

@torch.no_grad()
def evaluate(head, X, Y, bs=32):
    conf = torch.zeros(K * K, dtype=torch.long, device=DEV); ok, tot = [], []
    for i in range(0, len(Y), bs):
        y = Y[i:i + bs].to(DEV).long(); p = up(head(X[i:i + bs].to(DEV).float())).argmax(1); m = y != IGNORE
        conf += torch.bincount(y[m] * K + p[m], minlength=K * K)
        ok.append(((p == y) & m).flatten(1).sum(1).cpu()); tot.append(m.flatten(1).sum(1).cpu())
    return conf.view(K, K).cpu().numpy(), torch.cat(ok).numpy(), torch.cat(tot).numpy()

HEADS, CONF, IMG_ACC, TT = {}, {}, {}, {}
t0 = time.time(); print("Entrenando una cabeza por capa:", end=" ")
for l in LAYERS:
    t1 = time.time(); Xl, Yg = Xtr[l].to(DEV), Ytr.to(DEV)
    HEADS[l] = train_head(Xl, Yg, l); del Xl, Yg; torch.cuda.empty_cache()
    CONF[l], ok, tot = evaluate(HEADS[l], Xte[l], Yte); IMG_ACC[l] = (ok, tot); TT[l] = time.time() - t1; print(l, end=" · ", flush=True)
print(f"listo ({time.time() - t0:.0f} s).")

MAJ = int(np.argmax([int((Ytr == c).sum()) for c in range(K)]))
CONF["mayoritaria"] = np.zeros((K, K), np.int64); CONF["mayoritaria"][:, MAJ] = CONF[LAYERS[0]].sum(1)
log(f"Intervención: cabeza lineal idéntica en {len(LAYERS)} capas, semilla {CFG['seed']}, {CFG['epochs']} épocas, lr {CFG['lr']}. Referencia: siempre '{CLASS_NAMES[MAJ]}'.")

## 7. Resultados

Para cada capa se calcula el **IoU por clase** (intersección sobre unión entre lo predicho y lo real), el **mIoU** (promedio de los cuatro IoU) y la **exactitud** por píxel. Como métricas de eficiencia se reportan los **parámetros entrenables** (solo los de la cabeza), el costo analítico en `GFLOPs` de ejecutar el backbone hasta esa capa (truncar la red en un bloque intermedio reduciría el cómputo de forma lineal) y el tiempo de entrenamiento y evaluación de cada cabeza.

Al tratarse de una sola corrida, las diferencias menores que `tol` = 0,01 de mIoU se consideran ruido y no se interpretan. La referencia es la fila `mayoritaria`.

In [ ]:
def metrics(conf):
    tp = np.diag(conf).astype(float); gt = conf.sum(1); pr = conf.sum(0)
    iou = np.where(gt > 0, tp / np.maximum(gt + pr - tp, 1), np.nan)
    return float(np.nanmean(iou)), float(tp.sum() / conf.sum()), iou

name_of = lambda l: "mayoritaria" if l == "mayoritaria" else ("last_norm" if l == "last_norm" else f"bloque {l}")
RES = {l: dict(zip(("miou", "acc", "iou"), metrics(CONF[l]))) for l in ["mayoritaria"] + LAYERS}
TAB = pd.DataFrame({name_of(l): dict(mIoU=RES[l]["miou"], exactitud=RES[l]["acc"], **dict(zip(CLASS_NAMES, RES[l]["iou"])),
                                     GFLOPs=np.nan if l == "mayoritaria" else FL_BLOCK * (L if l == "last_norm" else l),
                                     **{"t. cabeza (s)": np.nan if l == "mayoritaria" else TT[l]})
                    for l in ["mayoritaria"] + LAYERS}).T
display(TAB.round(3)); TAB.round(4).to_csv(os.path.join(OUT_DIR, "resultados_e1.csv"))
N_HEAD = sum(p.numel() for p in HEADS[LAYERS[0]].parameters())
print(f"Parámetros entrenables por cabeza: {N_HEAD:,} · parámetros congelados del backbone: {n_params / 1e6:.1f} M")

BEST = max(LAYERS, key=lambda l: RES[l]["miou"]); BEST_NAME = name_of(BEST)
fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.2)); names = [str(l) for l in LAYERS]; vals = [RES[l]["miou"] for l in LAYERS]
cols = ["#c0392b" if l == "last_norm" else ("#1e8449" if l == BEST else "#2e86c1") for l in LAYERS]
ax[0].bar(names, vals, color=cols); ax[0].axhline(RES["mayoritaria"]["miou"], ls="--", c="gray", label="clase mayoritaria")
for i, v in enumerate(vals): ax[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax[0].set_xlabel("capa (bloque k o last_norm)"); ax[0].set_ylabel("mIoU en prueba"); ax[0].set_ylim(0, 1)
ax[0].set_title("mIoU por capa (verde: mejor · rojo: última capa)"); ax[0].legend(loc="lower right")
M = np.array([RES[l]["iou"] for l in LAYERS]); im = ax[1].imshow(M, cmap="viridis", vmin=0, vmax=1)
ax[1].set_xticks(range(K)); ax[1].set_xticklabels(CLASS_NAMES); ax[1].set_yticks(range(len(LAYERS))); ax[1].set_yticklabels(names); ax[1].set_title("IoU por clase y capa")
for i in range(len(LAYERS)):
    for j in range(K): ax[1].text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", color="w", fontsize=9)
plt.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "resultados_capas.png"), dpi=140); plt.show()

## 8. Contraste con las hipótesis

Se aplican las reglas de la sección 5. Dado que es una sola corrida, los veredictos describen lo observado **en esta corrida** y no se generalizan.

In [ ]:
tol = CFG["tol"]; miou = {l: RES[l]["miou"] for l in LAYERS}; inter = [l for l in LAYERS if l != "last_norm"]
best_i = max(inter, key=miou.get); gap = miou[best_i] - miou["last_norm"]
v1 = "se sostiene" if gap > tol else ("no se sostiene (last_norm es mejor)" if gap < -tol else "no concluyente (diferencia ≤ tol)")
v = np.array([miou[l] for l in LAYERS]); first = int(np.argmax(v >= v.max() - tol)); h2 = first > 0 and bool((v[first:] >= v.max() - tol).all())
rg = CLASS_NAMES.index("roca grande"); hardest = [CLASS_NAMES[int(np.nanargmin(RES[l]["iou"]))] for l in LAYERS]; h3 = all(c == "roca grande" for c in hardest)
by_acc = max(LAYERS, key=lambda l: RES[l]["acc"]); txt = lambda ok: "se sostiene" if ok else "no se sostiene"

VER = pd.DataFrame([
    ("H1", v1, f"mejor bloque intermedio: {best_i} (mIoU {miou[best_i]:.3f}) frente a last_norm ({miou['last_norm']:.3f}); diferencia {gap:+.3f}"),
    ("H1a", txt(h2), f"el mIoU alcanza su máximo (±{tol}) desde la capa {LAYERS[first]}; mIoU por capa: " + ", ".join(f"{name_of(l).replace('bloque ', 'b')} {miou[l]:.3f}" for l in LAYERS)),
    ("H1b", txt(h3), "clase de menor IoU en cada capa: " + ", ".join(sorted(set(hardest)))),
    ("Criterio", "mIoU aplicado", f"mejor en mIoU: {BEST_NAME}; mejor en exactitud: {name_of(by_acc)}" + (" (coinciden)" if by_acc == BEST else " (no coinciden)")),
], columns=["Hipótesis", "Veredicto", "Evidencia"]).set_index("Hipótesis")
display(VER)

dif = RES[BEST]["miou"] - RES["last_norm"]["miou"]
display(Markdown(f"**Mejor configuración (mIoU):** {BEST_NAME}, con mIoU {RES[BEST]['miou']:.3f} y exactitud {RES[BEST]['acc']:.3f}. "
                 f"**Última capa:** mIoU {RES['last_norm']['miou']:.3f} y exactitud {RES['last_norm']['acc']:.3f}. "
                 f"**Referencia (clase mayoritaria):** mIoU {RES['mayoritaria']['miou']:.3f}."
                 + (f" La diferencia entre la mejor configuración y la última capa ({dif:+.3f}) es menor que la tolerancia: no se interpreta como superioridad." if BEST != "last_norm" and abs(dif) <= tol else "")))
json.dump(dict(model=CFG["model_id"], n_train=len(TR), n_test=len(TE), seed=CFG["seed"], tol=tol, best=BEST_NAME,
               miou={str(l): RES[l]["miou"] for l in LAYERS}, acc={str(l): RES[l]["acc"] for l in LAYERS}, majority_miou=RES["mayoritaria"]["miou"],
               veredictos={h: r["Veredicto"] for h, r in VER.iterrows()}), open(os.path.join(OUT_DIR, "summary_e1.json"), "w"), indent=2, ensure_ascii=False)
log("Veredictos: " + "; ".join(f"{h} {r['Veredicto']}" for h, r in VER.iterrows()) + f". Mejor configuración: {BEST_NAME}.")

## 9. Dónde acierta y dónde falla la mejor configuración

Se muestran imágenes de **prueba** con cuatro paneles: la imagen, la **verdad** (etiquetas de expertos), la **predicción** de la mejor configuración y un **mapa de errores** (verde = acierto, rojo = error; solo en píxeles etiquetados). La figura de aciertos usa las imágenes con mejor exactitud entre las que contienen al menos dos clases; la de fallas, las de peor exactitud; la tercera, las imágenes con más roca grande, la clase más difícil.

In [ ]:
def predict(layer, i):
    with torch.no_grad(): return up(HEADS[layer](Xte[layer][i:i + 1].to(DEV).float())).argmax(1)[0].cpu().numpy()

def show_cases(idx, layer, title):
    fig, ax = plt.subplots(len(idx), 4, figsize=(14.5, 3.5 * len(idx)), squeeze=False)
    for r, i in enumerate(idx):
        y = Yte[i].numpy(); m = y != IGNORE; p = predict(layer, i)
        img = np.asarray(Image.open(TE[i][0]).convert("RGB").resize((S, S), RS.BILINEAR))
        pred = colorize(p); pred[~m] = 0.15
        err = np.full((S, S, 3), 0.15, np.float32); err[m & (p == y)] = (0.20, 0.70, 0.35); err[m & (p != y)] = (0.85, 0.15, 0.15)
        rock = 100 * (y == rg).sum() / max(m.sum(), 1)
        for a, im_, t in zip(ax[r], (img, colorize(y), pred, err),
                             (Path(TE[i][0]).name[:26], "verdad" + (f" · roca grande {rock:.1f} %" if rock > 0 else ""),
                              f"predicción · acierto {100 * (p[m] == y[m]).mean():.0f} %", "errores")):
            a.imshow(im_, interpolation="nearest"); a.set_title(t, fontsize=9); a.axis("off")
    fig.legend(handles=CLASS_LEGEND + [Patch(color=(0.20, 0.70, 0.35), label="acierto"), Patch(color=(0.85, 0.15, 0.15), label="error")],
               loc="lower center", ncol=7, frameon=False, fontsize=9)
    fig.suptitle(f"{title} · {BEST_NAME}", fontsize=12); plt.tight_layout(rect=(0, 0.04, 1, 0.97)); plt.show()

ok, tot = IMG_ACC[BEST]; acc_i = ok / np.maximum(tot, 1)
cls_px = np.stack([(Yte == c).flatten(1).sum(1).numpy() for c in range(K)], 1)
eligible = np.where(tot >= 0.05 * S * S)[0]
diverse = [i for i in eligible if (cls_px[i] >= 0.02 * tot[i]).sum() >= 2]
good = sorted(diverse, key=lambda i: -acc_i[i])[:3]; bad = sorted(eligible, key=lambda i: acc_i[i])[:3]
rocky = [i for i in np.argsort(-cls_px[:, rg])[:3] if cls_px[i, rg] > 0]
show_cases(good, BEST, "Aciertos"); show_cases(bad, BEST, "Fallas"); show_cases(rocky, BEST, "Imágenes con más roca grande")
log(f"Ejemplos cualitativos ({BEST_NAME}): aciertos {good}; fallas {bad}; roca grande {rocky} (índices de prueba).")

## 10. Análisis de fallas

La **matriz de confusión** (filas normalizadas) indica, de los píxeles reales de cada clase, qué fracción se predice como cada clase; la diagonal son los aciertos. Se compara la última capa con la mejor configuración, y la tabla lista las confusiones más frecuentes de esta última. Estas figuras muestran **qué** falla; explicar **por qué** (iluminación, distancia, ruido de etiquetas, ambigüedad entre clases) exige experimentos adicionales, por lo que cualquier explicación es una conjetura.

In [ ]:
CMP = BEST if BEST != "last_norm" else PRIMARY
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for a, l in zip(ax, ["last_norm", CMP]):
    Cm = CONF[l]; Cn = Cm / np.maximum(Cm.sum(1, keepdims=True), 1); a.imshow(Cn, cmap="Blues", vmin=0, vmax=1)
    for i in range(K):
        for j in range(K): a.text(j, i, f"{Cn[i, j]:.2f}", ha="center", va="center", color="w" if Cn[i, j] > 0.5 else "k")
    a.set_xticks(range(K)); a.set_xticklabels(CLASS_NAMES, rotation=20); a.set_yticks(range(K)); a.set_yticklabels(CLASS_NAMES)
    a.set_xlabel("predicho"); a.set_ylabel("real"); a.set_title(f"Confusión · {name_of(l)}")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "confusion.png"), dpi=140); plt.show()

Cm = CONF[CMP]; off = sorted(((Cm[i, j] / max(Cm[i].sum(), 1), i, j) for i in range(K) for j in range(K) if i != j), reverse=True)[:4]
display(pd.DataFrame([(CLASS_NAMES[i], CLASS_NAMES[j], f"{100 * v:.1f} %") for v, i, j in off], columns=["clase real", "predicha como", "% de la clase real"]).set_index("clase real"))
log(f"Confusiones más frecuentes ({name_of(CMP)}): " + "; ".join(f"{CLASS_NAMES[i]}→{CLASS_NAMES[j]} {100 * v:.1f} %" for v, i, j in off))

## 11. Decisiones de diseño, limitaciones y ética

### Decisiones de diseño y uso de asistentes de IA

| Decisión | Justificación | Asistencia de IA |
|---|---|---|
| Dominio y datos: AI4Mars (Curiosity); modelo principal DINOv3 ViT-B/16 (`lvd1689m`) | Definidos en la propuesta: dominio muy alejado del preentrenamiento y modelo base de la comparación | No |
| Pregunta de investigación y H1 | Tomadas de la propuesta | No |
| Hipótesis derivadas H1a y H1b y criterio de evaluación (mIoU) | Describen la forma de la curva por capa y el efecto del desbalance de clases | Redacción |
| Alcance preliminar: una semilla y reglas descriptivas con tolerancia 0,01 | Es una intervención preliminar; la tolerancia se fijó a partir de la variación entre semillas en pruebas piloto | Propuesta |
| Entrada de 448 px, 1 200 imágenes de entrenamiento, 20 épocas, capas 2 a 12 y `last_norm` | Cómputo limitado (GPU T4): resolución moderada y subconjunto de entrenamiento; las capas cubren la profundidad con siete puntos | Propuesta |
| Cabeza lineal (BatchNorm + convolución 1×1) sobre los patch tokens | Es el *linear probing* de la propuesta: aísla la información que ya contiene cada capa | Implementación |
| Entrenar con `labels/train` y evaluar en la prueba oficial, sin ajustar hiperparámetros con ella | Comparabilidad con la literatura y ausencia de fuga hacia la prueba | Propuesta |

El diseño y la redacción del código contaron con la asistencia de Claude (Anthropic). El equipo ejecutó el notebook, revisó los resultados y asume la responsabilidad de cada decisión. El notebook genera además `design_log.md`, un registro con hora de las decisiones y los resultados.

### Limitaciones y riesgos

- **Alcance:** una sola semilla, un solo subconjunto de entrenamiento, un solo modelo (ViT-B/16, `lvd1689m`), un solo rover y una sola cabeza lineal: **no se mide incertidumbre**, las diferencias menores que `tol` no son interpretables y los resultados no se generalizan.
- **Carácter preliminar:** las hipótesis se formularon tras una corrida piloto; este trabajo no es confirmatorio. Además, H1 compara el **mejor de seis bloques** con la última capa, lo que favorece a H1 frente a una comparación con un único bloque fijado de antemano.
- **Ruido de etiquetas:** el etiquetado colaborativo limita el techo de mIoU; la prueba usa etiquetas de expertos, de otro origen que las de entrenamiento.
- **Desbalance de clases:** la roca grande es ≈ 0,1 % de los píxeles de prueba; su IoU se estima con muy pocos píxeles, es muy ruidoso y, como el mIoU la pesa igual que las demás clases, arrastra el promedio. El reparto de clases de la prueba difiere del de entrenamiento, y parte del rendimiento puede deberse a ese desajuste y no a la capa evaluada.
- **Cómputo limitado (GPU T4):** modelo de tamaño B, resolución moderada (448 px), subconjunto de entrenamiento y sin aumentos de datos (las características se almacenan).
- **Acceso a los pesos:** DINOv3 es un modelo de acceso restringido, sujeto a la aceptación de su licencia.
- **Pendiente para la Entrega 3:** fine-tuning parcial (últimos k bloques y/o LoRA), curva de eficiencia por muestra etiquetada, comparación con DINOv3 `sat493m` y DINOv2, varias semillas con media ± desviación, y las hipótesis H2 y H3.

### Ética y licencias

Las imágenes son de superficie planetaria y no contienen personas ni datos personales. El conjunto **no debe emplearse para decisiones de navegación real** sin validación específica. Licencias: **AI4Mars** figura como CC BY 4.0 en Zenodo, Apache 2.0 en Kaggle y «no especificada» en NASA (se cita a Swan et al. y el registro de Zenodo, y se declara la discrepancia); **DINOv3** se distribuye bajo una licencia propia de Meta, aceptada al solicitar el acceso en Hugging Face.

### Trabajo relacionado inicial

DINOv3 (Siméoni et al., 2025, arXiv:2508.10104) y DINOv2 (Oquab et al., 2023, arXiv:2304.07193) son modelos de visión autosupervisados que se reutilizan congelados con cabezas ligeras. Alain y Bengio (2016) introdujeron las **sondas lineales** sobre capas intermedias, metodología que sigue este trabajo, y Raghu et al. (2021) compararon las representaciones de los ViT capa por capa. AI4Mars (Swan et al., 2021, CVPR Workshops) es el conjunto de datos utilizado. La revisión sistemática de la literatura queda para la entrega final.

### Reproducibilidad

Entorno: Kaggle con GPU T4, `torch` y `transformers` registrados en `env.json`; semilla fija (`CFG["seed"]`); unos 10–12 minutos de ejecución y ≈ 13 GB de RAM para las características. Requiere el dataset *AI4Mars - Mirrored Dataset* (v0.6) y acceso aprobado al modelo en Hugging Face (secreto `HF_TOKEN`). Salidas: `resultados_e1.csv`, `summary_e1.json`, `design_log.md` y las figuras.